# Single Subtitle — Burn

Third step of the pipeline (after `video-base-*.ipynb` and `caption-single-generate.ipynb`): burns the
chosen subtitle file onto the video base as a single, plain caption track.

**Which subtitle file gets used?** `config.nome_legenda_unica` — by default, the Whisper
transcription from `caption-single-generate.ipynb` (`{NOME}_whisper_{IDIOMA}.srt`). If you downloaded it,
corrected it, and re-uploaded it to Drive (same filename), this notebook picks up your
correction automatically — it always reads the current file from Drive, never a stale local
copy from an earlier session.

To use a **different** file instead (e.g. you saved your correction under a new name), set
`NOME_LEGENDA_UNICA` in the Configuration cell below.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📦 SETUP — packages, Google Drive, and modules (run once per session) ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── System packages ──────────────────────────────────────────────────────────
!apt-get -qq -y install ffmpeg fonts-noto-cjk > /dev/null 2>&1
print('✅ ffmpeg')

# ── Mount Drive (unmount first to avoid a stuck session) ────────────────────
from google.colab import drive
try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)
print('✅ Drive mounted')

# ── Copy modules from Drive to /content/pipeline ────────────────────────────
import shutil, os, sys, logging
from pathlib import Path

PASTA_DRIVE_RAIZ = "narrated_video"  # fixed for the whole project (same value as Configuration)
PASTA_MODULOS = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/pipeline/modulos")
DESTINO = Path("/content/pipeline")

if PASTA_MODULOS.exists():
    if DESTINO.exists():
        shutil.rmtree(DESTINO)
    shutil.copytree(PASTA_MODULOS, DESTINO)
    print(f"✅ {len(list(DESTINO.glob('*.py')))} modules copied from {PASTA_MODULOS}")

    # ── A cópia trouxe TODOS os módulos? ───────────────────────────────────────
    # "N módulos copiados" sozinho não quer dizer nada. E o modo de falhar aqui é
    # traiçoeiro: o Drive montado do Colab popula a listagem da pasta com atraso,
    # então um copytree logo depois do mount às vezes enxerga só parte dos
    # arquivos. Já aconteceu de copiar 13 de 31 -- com visto verde -- e o notebook
    # quebrar muito depois, num import, longe da causa.
    #
    # A conferência é de três pontas, porque a causa muda o conserto:
    #   manifesto  o que o repositório tem  (gravado pelo repositorio-sincronizar)
    #   Drive      o que chegou lá
    #   VM         o que a cópia desta célula trouxe
    # A conferência tem duas perguntas, e SÓ UMA delas precisa do manifesto:
    #
    #   Drive → VM   a cópia acima trouxe tudo?      dá pra ver aqui mesmo
    #   repo → Drive o Drive está em dia?            só o manifesto sabe
    #
    # A versão anterior amarrava as duas ao manifesto: sem ele, imprimia um
    # aviso e seguia SEM CONFERIR NADA. Foi assim que "✅ 13 modules copied"
    # passou com visto verde num Drive que tinha 31 -- justamente no dia em
    # que o manifesto ainda não existia. Comparar 13 com 31 nunca dependeu de
    # manifesto nenhum.
    _no_drive = {f.name for f in PASTA_MODULOS.glob("*.py")}
    _na_vm    = {f.name for f in DESTINO.glob("*.py")}

    # ── Drive → VM ────────────────────────────────────────────────────────
    # O Drive montado do Colab popula a listagem da pasta com atraso, então um
    # copytree logo depois do mount às vezes enxerga só parte dos arquivos.
    # Uma segunda passada, com o mount já quente, costuma resolver.
    _nao_copiados = sorted(_no_drive - _na_vm)
    if _nao_copiados:
        print(f"   ⏳ {len(_nao_copiados)} módulo(s) não vieram na 1ª passada — copiando de novo")
        for _n in _nao_copiados:
            shutil.copyfile(PASTA_MODULOS / _n, DESTINO / _n)
        _na_vm = {f.name for f in DESTINO.glob("*.py")}
        _nao_copiados = sorted(_no_drive - _na_vm)
    if _nao_copiados:
        print(f"\n🚨 {len(_nao_copiados)} módulo(s) estão no Drive mas não copiaram:")
        for _n in _nao_copiados:
            print(f"     {_n}")
        raise SystemExit("Rode ESTA célula de novo — o Drive montado ainda estava acordando.")
    print(f"   ✅ os {len(_no_drive)} módulos do Drive chegaram na VM")

    # ── repositório → Drive ───────────────────────────────────────────────
    _manifesto = PASTA_MODULOS / "_manifesto.txt"
    if not _manifesto.exists():
        print("   ⚠️  sem _manifesto.txt: não dá pra saber se o DRIVE está atrás")
        print("      do repositório. Rode o repositorio-sincronizar pra criá-lo.")
    else:
        _esperados = {l.strip() for l in _manifesto.read_text().splitlines()
                      if l.strip() and not l.startswith("#")}
        _fora_do_drive = sorted(_esperados - _no_drive)
        if _fora_do_drive:
            print(f"\n🚨 {len(_fora_do_drive)} módulo(s) não estão no DRIVE:")
            for _n in _fora_do_drive:
                print(f"     {_n}")
            raise SystemExit("Rode o repositorio-sincronizar.ipynb — o Drive está atrás do repositório.")
        print(f"   ✅ e batem com os {len(_esperados)} do manifesto")

    # ── O Python está segurando a versão anterior? ────────────────────────
    # Copiar arquivo novo por cima não desfaz um import já feito: o Python
    # guarda o módulo em sys.modules e reaproveita. Numa sessão longa, isso
    # faz o notebook rodar com o config.py de ontem mesmo depois de um sync
    # perfeito -- e o sintoma aparece longe da causa (nome de arquivo que
    # mudou, padrão que era pra ter mudado e não mudou). Descarregar aqui
    # equivale a reiniciar o runtime, sem perder o resto da sessão.
    _recarregar = [_n for _n, _m in list(sys.modules.items())
                   if getattr(_m, "__file__", None) and str(DESTINO) in str(_m.__file__)]
    for _n in _recarregar:
        del sys.modules[_n]
    if _recarregar:
        print(f"   ♻️  {len(_recarregar)} módulo(s) já importados foram descarregados —")
        print(f"      o import vai reler a cópia nova (rode as células seguintes de novo)")
else:
    print(f"❌ Modules folder not found: {PASTA_MODULOS}")
    print("   Make sure the .py files are in pipeline/modulos/ on Drive.")

if str(DESTINO) not in sys.path:
    sys.path.insert(0, str(DESTINO))

logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(name)-18s  %(levelname)s  %(message)s', datefmt='%H:%M:%S')
os.chdir('/content')
print('✅ Setup complete!')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⚙️  CONFIGURATION                                               ║
# ║  ✏️  Edit only this cell — same NOME_ORACAO as the earlier steps ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── 1. VIDEO IDENTITY (must match video-base-*.ipynb / caption-single-generate.ipynb) ──
NOME_ORACAO = "40_Matt_02"

# ── 2. NARRATION LANGUAGE (must match caption-single-generate.ipynb) ────────────────
IDIOMA_MESTRE = "en"

# ── 3. WHICH SUBTITLE FILE TO BURN ──────────────────────────────────────────
# Leave blank to use the default (the Whisper transcription from
# caption-single-generate.ipynb: {NOME}_whisper_{IDIOMA}.srt). Fill in only if you want
# to point at a different SRT already saved in the video's folder on Drive
# (e.g. a manual correction saved under a new name).
NOME_LEGENDA_UNICA = ""

# ── 4. VERSE REFERENCE OVERLAY (optional) ───────────────────────────────────
# A small fixed indicator in the top-left corner (e.g. "Matt/Mt/마 2:4") that
# updates only the verse number as the narration advances — separate from
# the subtitle itself. OPTIONAL: only makes sense for verse-by-verse Bible
# study videos — leave INCLUIR_VERSICULO = False for prayer/free-content
# videos that aren't tied to Bible verses.
INCLUIR_VERSICULO = True

# Only used if INCLUIR_VERSICULO = True:

# CAPITULO: leave None to derive it from NOME_ORACAO — "40_Matt_02" already
# says the chapter is 2. The old default was a hard-coded 1, which on a video
# named 40_Matt_02 burned "Matt 1:4" over chapter 2 with no error at all. Only
# fill this in for a video whose name does NOT follow {NN}_{Sigla}_{CC}.
CAPITULO = None

ABREVIACOES_LIVRO = {"en": "Matt", "pt": "Mt", "es": "Mt", "fr": "Mt", "ko": "마"}

# TEXTO_VERSICULOS: leave "" and the notebook finds the text itself — the
# video's roteiro_versiculos.txt on Drive, or the whole Bible in
# dados_lexico/web-biblia.json. Fill this in only to use a text different
# from both (verse numbers as standalone tokens in the flow:
# "1 Now when Jesus was born ... 2 Where is he who is born ...").
TEXTO_VERSICULOS = ""

# ── 5. DRIVE ROOT FOLDER ──────────────────────────────────────────────────
PASTA_DRIVE_RAIZ = "narrated_video"     # ⚠️ DO NOT CHANGE — fixed for the whole project

# ── CHECK ──────────────────────────────────────────────────────────────────
print("=" * 60)
print("⚙️  CONFIGURATION")
print("=" * 60)
print(f"   Video:           {NOME_ORACAO}")
print(f"   Narration lang:  {IDIOMA_MESTRE}")
print(f"   Subtitle file:   {NOME_LEGENDA_UNICA or '(default — Whisper transcription)'}")
print(f"   Verse overlay:   {'ON — ' + '/'.join(dict.fromkeys(ABREVIACOES_LIVRO.values())) + f' {CAPITULO}:N' if INCLUIR_VERSICULO else 'off'}")
print(f"   Drive root:      {PASTA_DRIVE_RAIZ}")
print("=" * 60)
print("✅ Configuration ready — proceed to Initialization")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🚀 INITIALIZE PIPELINE                                          ║
# ╚══════════════════════════════════════════════════════════════════╝

import sys
from pathlib import Path

if '/content/pipeline' not in sys.path:
    sys.path.insert(0, '/content/pipeline')

from config import PipelineConfig
from caption_pipeline import CaptionPipeline

# O nome do vídeo já carrega livro e capítulo — não peça duas vezes o que
# `40_Matt_02` só pode significar. Recusa nome fora do padrão em vez de
# adivinhar: capítulo adivinhado errado não dá erro, só sai no vídeo.
if CAPITULO is None:
    import biblia_livros as _bl
    _livro, CAPITULO = _bl.de_nome_projeto(NOME_ORACAO)
    print(f"📖 Chapter derived from {NOME_ORACAO}: {_livro.nome} {CAPITULO}")


# Fundo: "imagem", "video", ou None pra detectar sozinho pelo arquivo que
# existe no Drive. Só preencha à mão se as duas versões estiverem lá.
MODO_CLIPE = None

# O modo do fundo (imagem parada ou clipe de vídeo) sai do arquivo que EXISTE
# no Drive, não de uma opção que dá pra esquecer de marcar -- mesma ideia do
# sufixo `_zh` lido do nome do .ass. Sem isto, queimar sobre um vídeo base
# feito em modo imagem procurava `_video_base.mp4` e não achava; e se achasse,
# o resultado sairia sem `_img`, por cima da versão de clipe.
import config as _cfgmod
if MODO_CLIPE is None:
    MODO_CLIPE = _cfgmod.detectar_modo_clipe(
        Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/videos/{NOME_ORACAO}"),
        NOME_ORACAO,
    )
    print(f"🎞️  Background mode detected from Drive: {MODO_CLIPE}")

config = PipelineConfig(
    NOME_ORACAO         = NOME_ORACAO,
    PASTA_DRIVE_RAIZ     = PASTA_DRIVE_RAIZ,
    IDIOMA_MESTRE         = IDIOMA_MESTRE,
    NOME_LEGENDA_UNICA    = NOME_LEGENDA_UNICA,
    MODO_CLIPE            = MODO_CLIPE,
    CAPITULO              = CAPITULO,
    ABREVIACOES_LIVRO     = ABREVIACOES_LIVRO,
)

pipeline = CaptionPipeline(config)

print("=" * 60)
print("✅ PIPELINE INITIALIZED")
print("=" * 60)
print(f"   Video:           {config.NOME_ORACAO}")
print(f"   Folder:          {config.pasta_oracao}")
print(f"   Subtitle file:   {config.nome_legenda_unica}")
print(f"   Output video:    {config.NOME_VIDEO_FINAL}")
print("=" * 60)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📖 GENERATE VERSE REFERENCE (optional — only if INCLUIR_VERSICULO) ║
# ║  Skipped automatically if INCLUIR_VERSICULO = False.              ║
# ╚══════════════════════════════════════════════════════════════════╝

if INCLUIR_VERSICULO:
    # Três fontes, da mais específica pra mais geral (ver
    # caption_pipeline.resolver_texto_versiculos): o que você colou acima, o
    # roteiro do vídeo no Drive, ou o web-biblia.json com a Bíblia inteira.
    from caption_pipeline import resolver_texto_versiculos

    TEXTO_VERSICULOS, _origem = resolver_texto_versiculos(config, TEXTO_VERSICULOS)
    print(f"📖 Verse text from {_origem} ({len(TEXTO_VERSICULOS)} chars)")

    srt_versiculo = pipeline.gerar_legenda_versiculo(TEXTO_VERSICULOS)
    print(f"✅ Verse reference generated: {srt_versiculo.name}")
else:
    print("Verse reference overlay is off (INCLUIR_VERSICULO = False) — skipping.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔥 LOAD + BURN — single subtitle track onto the video base      ║
# ║  Always downloads the current file from Drive — picks up any     ║
# ║  manual correction automatically.                                 ║
# ╚══════════════════════════════════════════════════════════════════╝

legendas = pipeline.carregar_legenda_unica()
print(f"{len(legendas)} caption blocks loaded from {config.nome_legenda_unica}\n")

video_final = pipeline.queimar_legenda_unica(legendas, incluir_versiculo=INCLUIR_VERSICULO)
print(f"\n✅ Final video: {video_final.name} ({video_final.stat().st_size/1_048_576:.1f} MB)")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  👀 PREVIEW FINAL VIDEO                                          ║
# ╚══════════════════════════════════════════════════════════════════╝

from IPython.display import Video, display

display(Video(str(video_final), embed=True, width=800))


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📥 DOWNLOAD — FINAL VIDEO                                       ║
# ╚══════════════════════════════════════════════════════════════════╝

from google.colab import files

print(f"📥 Downloading {video_final.name} ({video_final.stat().st_size/1_048_576:.1f} MB)...")
files.download(str(video_final))
